In [7]:

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


import torch.multiprocessing as mp
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.distributed import init_process_group, destroy_process_group
import os
import xarray as xr
import numpy as np
import glob
import random

import sys


In [3]:


def ddp_setup():
    torch.cuda.set_device(int(os.environ["LOCAL_RANK"]))
    init_process_group(backend="nccl")

In [132]:
from mlp_training.mlp import MLP

In [139]:
import torch

In [134]:

dd = MLP(60*4+4, 60*4+8)

In [144]:
saved = torch.load('/glade/work/qingyuany/Climsim/ml_results/mlp_training/test.pt')

saved['MODEL_STATE']

OrderedDict([('network.0.weight',
              tensor([[-0.0470, -0.0608,  0.0023,  ...,  0.0115, -0.0022,  0.0153],
                      [-0.0400,  0.0172,  0.0316,  ..., -0.0092,  0.0188, -0.0361],
                      [ 0.0208,  0.0325, -0.0259,  ...,  0.0546,  0.0464,  0.0483],
                      ...,
                      [ 0.0452,  0.0167,  0.0423,  ..., -0.0277, -0.0460,  0.0417],
                      [-0.0391, -0.0902, -0.0102,  ..., -0.0150,  0.0281,  0.0595],
                      [-0.0110, -0.0413, -0.1077,  ...,  0.0737,  0.1003,  0.0964]],
                     device='cuda:0')),
             ('network.0.bias',
              tensor([-0.0025, -0.0472,  0.0601,  0.1017,  0.0258,  0.1065,  0.0758,  0.0100,
                       0.0304,  0.0329,  0.0805, -0.0286,  0.0797, -0.0046,  0.0147,  0.0555,
                       0.0987,  0.0559,  0.0777,  0.0532,  0.0200, -0.0476,  0.0803, -0.0030,
                       0.0153,  0.0559,  0.0679,  0.0182,  0.0777, -0.0102,  0.0

In [145]:
dd.load_state_dict(saved['MODEL_STATE'])

<All keys matched successfully>

In [16]:

inp_var_3d_nm = ['state_t', 'state_q0001', 'state_q0002', 'state_q0003']
inp_var_2d_nm = ['state_ps', 'pbuf_SOLIN', 'pbuf_LHFLX', 'pbuf_SHFLX']
                 

out_var_3d_nm = ['state_t', 'state_q0001', 'state_q0002', 'state_q0003']
out_var_2d_nm = ['cam_out_NETSW',  
                 'cam_out_FLWDS', 'cam_out_PRECSC', 'cam_out_PRECC', 'cam_out_SOLS', 
                 'cam_out_SOLL', 'cam_out_SOLSD', 'cam_out_SOLLD']

########################################################
xynames = [inp_var_3d_nm, inp_var_2d_nm, out_var_3d_nm, out_var_2d_nm]

inp_files_raw = glob.glob("/glade/work/qingyuany/Climsim/train/*/E3SM*mli*.nc")
out_files_raw = glob.glob("/glade/work/qingyuany/Climsim/train/*/E3SM*mlo*.nc")
random.shuffle(inp_files_raw)
inp_files_raw = inp_files_raw[:5]

inp_files = []
count = 0
for f in inp_files_raw:
    temp = f.replace("mli", "mlo")
    if temp in out_files_raw:
        inp_files.append(f)

    else:
        count += 1

In [39]:
from dataset.dataset import (Climsim_Dataset_xy, transform23d, transform_out, transform_q, 
        name_mapping, load_norm_mean_std, load_lambda, ClimsimBaseDataset, Climsim_Dataset_xy)

In [40]:
def load_train_objs(inp_files, inp_var_3d_nm, inp_var_2d_nm, out_var_3d_nm, out_var_2d_nm):
    dataset = Climsim_Dataset_xy(
        input_paths = inp_files,
        xname3d = inp_var_3d_nm,
        xname2d = inp_var_2d_nm,
        yname3d = out_var_3d_nm,
        yname2d = out_var_2d_nm       
    )   
    model = MLP(input_dim = len(inp_var_3d_nm) * 60 + len(inp_var_2d_nm), output_dim = len(out_var_3d_nm) * 60 + len(out_var_2d_nm))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    return dataset, model, optimizer





In [49]:


def prepare_dataloader(dataset: Dataset, batch_size: int):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=4,
        pin_memory=True
    )


In [128]:
    
class Trainer:
    def __init__(self, dataloader, model, optimizer, loss_fun, bsz, xynames, save_every, snapshot_path):

        self.dataloader = dataloader
        self.model = model
        self.optimizer = optimizer
        self.loss_fun = loss_fun
        self.bsz = bsz
        self.save_every = save_every
        self.snapshot_path = snapshot_path
        self.gpu_id = 0
        self.x3d_name, self.x2d_name, self.y3d_name, self.y2d_name = xynames
        self.epochs_run = 0
        #self.gpu_id = int(os.environ["LOCAL_RANK"])
        self.model = model.to(self.gpu_id)
        device_ids=[self.gpu_id], 0
    
        if os.path.exists(snapshot_path):
            print("Loading snapshot")
            self._load_snapshot(snapshot_path)

    
    def _load_snapshot(self, snapshot_path):
        loc = f"cuda:{self.gpu_id}"
        snapshot = torch.load(snapshot_path, map_location=loc)
        self.model.load_state_dict(snapshot["MODEL_STATE"])
        self.epochs_run = snapshot["EPOCHS_RUN"]
        print(f"Resuming training from snapshot at Epoch {self.epochs_run}")

    def _save_snapshot(self, epoch):
        snapshot = {
            "MODEL_STATE": self.model.state_dict(),
            "EPOCHS_RUN": epoch,
        }
        torch.save(snapshot, self.snapshot_path)
        print(f"Epoch {epoch} | Training snapshot saved at {self.snapshot_path}")


    def _run_batch(self, x_batch, y_batch):
        self.optimizer.zero_grad()
        print(x_batch.shape)
        y_pred = self.model(x_batch)
        loss = self.loss_fun(y_pred, y_batch)
        loss.backward()
        self.optimizer.step()
        return loss.item()
    
    def _run_epoch(self, epoch):
        for x3d, x2d, y3d, y2d in self.dataloader:
            x3d = x3d.reshape(-1, len(self.x3d_name) * 60)
            x2d =x2d.reshape(-1, len(self.x2d_name))

            y3d = y3d.reshape(-1, len(self.y3d_name) * 60)
            y2d = y2d.reshape(-1, len(self.y2d_name))

            x = torch.cat([x3d, x2d], dim=1).to(self.gpu_id)    
            y = torch.cat([y3d, y2d], dim=1).to(self.gpu_id)

    
            loss = self._run_batch(x, y)

    def train(self, max_epochs: int):
        for epoch in range(self.epochs_run, max_epochs):
            self._run_epoch(epoch)
            if self.gpu_id == 0 and epoch % self.save_every == 0:
                self._save_snapshot(epoch)

In [107]:
dataset, model, optimizer = load_train_objs(inp_files, inp_var_3d_nm, inp_var_2d_nm, out_var_3d_nm, out_var_2d_nm)




In [110]:
model

MLP(
  (network): Sequential(
    (0): Linear(in_features=244, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=248, bias=True)
  )
)

In [113]:
model(torch.randn(244, 244))

tensor([[-0.1930,  0.1409,  0.0665,  ...,  0.1000,  0.1005, -0.0653],
        [-0.1604,  0.2169,  0.1736,  ...,  0.0927,  0.1069, -0.0461],
        [-0.0951,  0.1998,  0.1607,  ...,  0.1580, -0.0642, -0.0894],
        ...,
        [-0.1059,  0.2390,  0.2113,  ...,  0.1256,  0.2213,  0.0292],
        [-0.1240,  0.2472,  0.2569,  ...,  0.1941,  0.2079, -0.0768],
        [-0.0600,  0.2346,  0.2264,  ...,  0.1746,  0.0749, -0.1309]],
       grad_fn=<AddmmBackward0>)

In [96]:
train_dataloader = prepare_dataloader(dataset, 2)

In [38]:
dataset = Climsim_Dataset_xy(
    input_paths = inp_files,
    xname3d = inp_var_3d_nm,
    xname2d = inp_var_2d_nm,
    yname3d = out_var_3d_nm,
    yname2d = out_var_2d_nm       
)   

In [77]:
loss_fun = torch.nn.MSELoss()
batch_size = 3
xynames = [inp_var_3d_nm, inp_var_2d_nm, out_var_3d_nm, out_var_2d_nm]

In [129]:
trainer = Trainer(train_dataloader, model, optimizer, loss_fun, batch_size, xynames, save_every = 1, snapshot_path= './data.pt')





In [130]:
trainer.train(2)

torch.Size([768, 244])
torch.Size([768, 244])
torch.Size([384, 244])
Epoch 0 | Training snapshot saved at ./data.pt
torch.Size([768, 244])
torch.Size([768, 244])
torch.Size([384, 244])
Epoch 1 | Training snapshot saved at ./data.pt
